<a href="https://colab.research.google.com/github/C-1Blue/ds2002-fa26/blob/main/2026_09_25_%E2%80%94_Cleaning_Gauntlet_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [15]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
rows_affected = before - after
log("drop duplicates", "removed duplicate rows", rows_affected)

[drop duplicates] removed duplicate rows (14 row(s))


### TODO 2 — clean `price` -> float

In [4]:
df['price'] = df['price'].astype(str).str.replace('$','', regex=False).astype(float)
df.head()
log("clean price", "Stipped $ and whitespace and convert to float", len(df))

[clean price] Stipped $ and whitespace and convert to float (315 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
# TODO
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')
log("convert qty to numeric", "convert qty to numeric", len(df))
missingqty = df['qty'].isna().sum
df = df[~df['qty'].isna()]
log("drop missing qty", "drop missing qty", missingqty)
negativeqty = (df['qty'] < 0).sum()
df = df[df['qty'] >= 0]
log("drop negative qty", "drop negative qty", negativeqty)

[convert qty to numeric] convert qty to numeric (315 row(s))
[drop missing qty] drop missing qty (<bound method Series.sum of 0      False
1      False
2      False
3       True
4      False
       ...  
266    False
147     True
299    False
58     False
122    False
Name: qty, Length: 315, dtype: bool> row(s))
[drop negative qty] drop negative qty (13 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [11]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(df['item'].value_counts())
variants = sorted(df['item'].unique())
print(variants)
ITEM_MAP = {'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',

    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',

    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho'
}
before = df['item'].nunique()
df['item'] = df['item'].map(ITEM_MAP)
after = df['item'].nunique()
log("normalize item", "collpase item spellings from variant before to after canon", before - after)

item
Foam Finger      57
Rain Poncho      51
cheese burger    47
rain poncho      46
Cheeseburger     46
foam finger      42
Name: count, dtype: int64
[np.str_('Cheeseburger'), np.str_('Foam Finger'), np.str_('Rain Poncho'), np.str_('cheese burger'), np.str_('foam finger'), np.str_('rain poncho')]
[normalize item] collpase item spellings from variant before to after canon (3 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [12]:
# TODO
print(df['category'].value_counts())
variants = sorted(df['category'].unique())
print(variants)
CATEGORY_MAP = {'Food': 'Food',
    'food': 'Food',

    'Merch': 'Merch',
    'Apparel': 'Merch',
    'RainGear': 'Merch',
    'rain-gear': 'Merch'
}
before = df['category'].nunique()
df['category'] = df['category'].map(CATEGORY_MAP)
after = df['category'].nunique()
log("normalize category", "collpase category spellings from variant before to after canon- business rule applied: apparel -> merch", before - after)

category
Food         54
Merch        54
food         48
rain-gear    47
Apparel      44
RainGear     42
Name: count, dtype: int64
[np.str_('Apparel'), np.str_('Food'), np.str_('Merch'), np.str_('RainGear'), np.str_('food'), np.str_('rain-gear')]
[normalize category] collpase category spellings from variant before to after canon- business rule applied: apparel -> merch (4 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [17]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
# TODO: assert something about item
EXPECTED_ITEMS = {"Cheeseburger", "Foam Finger", "Rain Poncho"}
assert set(df['item']) == EXPECTED_ITEMS, "Regression: unexpected item values detected"

# TODO: assert something about category
EXPECTED_CATEGORIES = {"Food", "Merch"}
assert set(df['category']) == EXPECTED_CATEGORIES, "Regression: unexpected category values detected"
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [19]:
# TODO
df['revenue'] = df['qty'] * df['price']
rev_by_cat = (
    df.groupby('category', as_index=False)
      .agg(total_revenue=('revenue','sum'))
)

rev_by_cat['total_revenue'] = rev_by_cat['total_revenue'].round(2)

rev_by_cat = rev_by_cat.sort_values('total_revenue', ascending=False)

print(rev_by_cat)
overall_total = df['revenue'].sum().round(2)
print("Overall total revenue:", overall_total)


  category  total_revenue
1    Merch         3084.0
0     Food         1656.0
Overall total revenue: 4740.0


/tmp/ipykernel_6192/461379748.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['revenue'] = df['qty'] * df['price']


**What I would tell the vendor:** _To stock more merch since it makes the highest revenue_

### TODO 8 — read back your log

In [20]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,315,order_id item category qty ...,0
1,clean price,Stipped $ and whitespace and convert to float,315
2,convert qty to numeric,convert qty to numeric,315
3,drop missing qty,drop missing qty,<bound method Series.sum of 0 False\n1 ...
4,drop negative qty,drop negative qty,13
5,normalize item,collpase item spellings from variant before to...,3
6,normalize category,collpase category spellings from variant befor...,4
7,drop duplicates,removed duplicate rows,14


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

_